# 1. CNN from Scratch

In [1]:
import pandas as pd

# read csv
preds = pd.read_csv('cnn_scratch_predictions.csv', index_col=0)
preds.head()

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2
0,27.627766,96.078100,37.559749,126.940378,"74-23 Sinchon-dong, Seodaemun-gu, Seoul, South...",KR
1,48.381645,7.905346,58.350841,15.562212,"702, 585 97 Linköping, Sweden",SE
2,50.516630,8.096517,42.195500,-7.728960,"OU-0110, 32678 Quintela, Province of Ourense, ...",ES
3,35.348820,5.560653,-24.601419,25.925948,"Airport Road, Gaborone, Botswana",BW
4,51.787643,6.671671,57.239148,26.494123,"P34, Lejasciema pagasts, LV-4412, Latvia",LV


In [2]:
# function to find haversine distance between two points on the earth
import math
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return c * 6371

# create new column in df for haversine distance
preds['distance'] = preds.apply(lambda row: haversine(row['pred_latitude'], row['pred_longitude'], row['true_latitude'], row['true_longitude']), axis=1)
preds.head()

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2,distance
0,27.627766,96.078100,37.559749,126.940378,"74-23 Sinchon-dong, Seodaemun-gu, Seoul, South...",KR,3077.749358
1,48.381645,7.905346,58.350841,15.562212,"702, 585 97 Linköping, Sweden",SE,1217.572403
2,50.516630,8.096517,42.195500,-7.728960,"OU-0110, 32678 Quintela, Province of Ourense, ...",ES,1521.539125
3,35.348820,5.560653,-24.601419,25.925948,"Airport Road, Gaborone, Botswana",BW,7002.377821
4,51.787643,6.671671,57.239148,26.494123,"P34, Lejasciema pagasts, LV-4412, Latvia",LV,1409.019520


In [3]:
preds['distance'].describe()

count     6633.000000
mean      4745.831195
std       4494.918832
min         10.938648
25%       1057.215586
50%       2010.500068
75%       8782.387324
max      19115.447287
Name: distance, dtype: float64

In [4]:
# The Geoguessr score takes the form of an exponential function given by score = 5000 * e ^ (-10 * distance / size)  
# where size = 14,916,862 meters for the world map.

# add a column for the score
preds['score'] = 5000 * (math.e ** (-10 * preds['distance'] / 14916.862))
preds.head()

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2,distance,score
0,27.627766,96.078100,37.559749,126.940378,"74-23 Sinchon-dong, Seodaemun-gu, Seoul, South...",KR,3077.749358,635.190244
1,48.381645,7.905346,58.350841,15.562212,"702, 585 97 Linköping, Sweden",SE,1217.572403,2210.456257
2,50.516630,8.096517,42.195500,-7.728960,"OU-0110, 32678 Quintela, Province of Ourense, ...",ES,1521.539125,1802.951493
3,35.348820,5.560653,-24.601419,25.925948,"Airport Road, Gaborone, Botswana",BW,7002.377821,45.737712
4,51.787643,6.671671,57.239148,26.494123,"P34, Lejasciema pagasts, LV-4412, Latvia",LV,1409.019520,1944.210904


In [5]:
preds['score'].describe()

count    6633.000000
mean     1361.115992
std      1353.129525
min         0.013603
25%        13.868725
50%      1299.051376
75%      2461.323749
max      4963.468727
Name: score, dtype: float64

# 2. Fine-tuned CNN

In [6]:
preds2 = pd.read_csv('cnn_finetune_predictions.csv', index_col=0)
preds2['distance'] = preds2.apply(lambda row: haversine(row['pred_latitude'], row['pred_longitude'], row['true_latitude'], row['true_longitude']), axis=1)
preds2['score'] = 5000 * (math.e ** (-10 * preds2['distance'] / 14916.862))
preds2.head()

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2,distance,score
0,47.614910,65.313660,37.559749,126.940378,"74-23 Sinchon-dong, Seodaemun-gu, Seoul, South...",KR,5029.860556,171.616749
1,57.823612,12.500967,58.350841,15.562212,"702, 585 97 Linköping, Sweden",SE,189.231426,4404.296392
2,45.026240,7.554272,42.195500,-7.728960,"OU-0110, 32678 Quintela, Province of Ourense, ...",ES,1267.931773,2137.076804
3,16.361317,47.871155,-24.601419,25.925948,"Airport Road, Gaborone, Botswana",BW,5139.148602,159.492838
4,53.530365,18.913162,57.239148,26.494123,"P34, Lejasciema pagasts, LV-4412, Latvia",LV,631.320021,3274.653807


In [7]:
preds2['distance'].describe()

count     6633.000000
mean      4143.832505
std       4157.044614
min         21.722107
25%        979.537584
50%       2209.217593
75%       6708.218237
max      18806.137997
Name: distance, dtype: float64

In [8]:
preds2['score'].describe()

count    6633.000000
mean     1438.950227
std      1385.422190
min         0.016737
25%        55.707932
50%      1137.027703
75%      2592.890485
max      4927.716995
Name: score, dtype: float64

In [9]:
# view the largest distances
preds2.sort_values('distance', ascending=False).head(20)

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2,distance,score
32,37.074097,8.300976,-40.755893,175.135088,"14 Matai Street, Otaki 5512, New Zealand",NZ,18806.137997,0.016737
118,51.114690,4.074992,-43.391946,172.651402,"9 Isaac Wilson Road, Kaiapoi 7630, New Zealand",NZ,18800.847542,0.016797
1612,42.166490,7.947325,-37.108699,175.090895,"353-355 Gelling Road, Hunua 2583, Nowa Zelandia",IS,18780.663083,0.017026
195,44.747044,8.579947,-38.103766,175.725925,"743 Old Taupo Road, Waotu 3481, New Zealand",NZ,18715.917131,0.017781
497,46.525524,9.555347,-40.231998,175.733189,"720 Watershed Road, Bunnythorpe 4470, New Zealand",IS,18699.726895,0.017975
843,28.252693,122.035230,-33.479625,-70.578798,"Vespucio Sur Express, Macul, Santiago Metropol...",CL,18679.536867,0.018220
1839,45.100060,9.249949,-37.108699,175.090895,"353-355 Gelling Road, Hunua 2583, Nowa Zelandia",NZ,18536.434943,0.020054
280,33.666374,0.255489,-45.255057,170.268966,"9750 Kyeburn-Hyde Road, Tiroiti 9397, New Zealand",NZ,18470.046840,0.020967
1723,42.554363,9.062125,-35.306215,173.649868,"2637 State Highway 1, Horeke 0475, New Zealand",NZ,18460.863862,0.021097
959,34.236008,11.927733,-35.759569,174.505432,"195 Pataua South Road, Pataua 0192, New Zealand",NZ,18421.233691,0.021665


In [10]:
preds.sort_values('distance', ascending=False).head(20)

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2,distance,score
1194,40.948883,5.801743,-38.228788,175.906381,"247-321 Mossop Road, Kinleith 3491, New Zealand",NZ,19115.447287,0.013603
909,46.450760,5.533533,-43.550697,172.633730,"219 Milton Street, Sydenham, Christchurch 8023...",NZ,18952.237532,0.015176
888,44.098236,4.478706,-45.669449,170.637637,"824 Coast Road, Seacliff 9471, New Zealand",NZ,18912.139756,0.015589
22,45.683323,7.164390,-41.190386,174.954750,"429-430 Cambridge Terrace, Naenae, Lower Hutt ...",NZ,18911.510303,0.015596
1270,45.573090,5.652292,-43.461736,172.024498,"182 Deans Road, Darfield 7571, New Zealand",NZ,18910.839680,0.015603
1273,46.518710,6.666740,-43.391946,172.651402,"9 Isaac Wilson Road, Kaiapoi 7630, New Zealand",NZ,18860.575467,0.016138
118,47.082760,6.611649,-43.391946,172.651402,"9 Isaac Wilson Road, Kaiapoi 7630, New Zealand",NZ,18849.489099,0.016258
704,44.755950,5.545860,-45.669449,170.637637,"824 Coast Road, Seacliff 9471, New Zealand",NZ,18844.560903,0.016312
567,43.896490,6.631114,-43.461736,172.024498,"182 Deans Road, Darfield 7571, New Zealand",NZ,18840.984497,0.016351
726,45.641266,7.149576,-38.471716,176.355704,"171 Strathmore Road, Reporoa 3081, New Zealand",NZ,18821.321701,0.016568


# 3. Fine-tuned Transformer

In [11]:
preds3 = pd.read_csv('finetune_predictions.csv', index_col=0)
preds3['distance'] = preds3.apply(lambda row: haversine(row['pred_latitude'], row['pred_longitude'], row['true_latitude'], row['true_longitude']), axis=1)
preds3['score'] = 5000 * (math.e ** (-10 * preds3['distance'] / 14916.862))
preds3.head()

,pred_latitude,pred_longitude,true_latitude,true_longitude,address,country_iso_alpha2,distance,score
0,45.771870,58.735540,37.559749,126.940378,"74-23 Sinchon-dong, Seodaemun-gu, Seoul, South...",KR,5565.376608,119.852679
1,49.352642,12.530680,58.350841,15.562212,"702, 585 97 Linköping, Sweden",SE,1019.853711,2523.750410
2,43.846706,24.926857,42.195500,-7.728960,"OU-0110, 32678 Quintela, Province of Ourense, ...",ES,2643.813264,849.652651
3,16.870628,13.996082,-24.601419,25.925948,"Airport Road, Gaborone, Botswana",BW,4789.481694,201.624997
4,55.195786,24.891087,57.239148,26.494123,"P34, Lejasciema pagasts, LV-4412, Latvia",LV,247.871375,4234.517262


In [12]:
preds3['distance'].describe()

count     6633.000000
mean      3971.451048
std       4121.410447
min         16.704126
25%       1036.770234
50%       2056.417094
75%       6068.056513
max      19883.040551
Name: distance, dtype: float64

In [13]:
preds3['score'].describe()

count    6633.000000
mean     1444.934592
std      1327.740956
min         0.008131
25%        85.564940
50%      1259.673206
75%      2495.291401
max      4944.321578
Name: score, dtype: float64